In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import time
import os
import gc

In [2]:
def is_english(text):
    if not isinstance(text, str):
        return False
    if len(text.strip()) == 0:
        return False
    
    # Kiểm tra tỷ lệ ký tự ASCII để loại bỏ tiếng Trung, Nhật, v.v.
    try:
        text.encode('ascii')
        is_ascii = True
    except UnicodeEncodeError:
        is_ascii = False
        
    if not is_ascii:
        ascii_chars = sum(1 for c in text if ord(c) < 128)
        if ascii_chars / len(text) < 0.9:
            return False
            
    # Kiểm tra sự xuất hiện của các từ dừng tiếng Anh phổ biến để đảm bảo text là tiếng Anh
    common_words = {'the', 'and', 'a', 'of', 'to', 'is', 'in', 'it', 'you', 'that', 'this', 'for', 'with', 'on'}
    words = set(text.lower().split())
    if len(words) > 5:
        if not words.intersection(common_words):
            return False
            
    return True

def text_generator(file_path, chunk_size=100000):
    for chunk in pd.read_csv(file_path, usecols=['content'], chunksize=chunk_size):
        for val in chunk['content'].fillna(''):
            val_str = str(val)
            if is_english(val_str):
                yield val_str

In [3]:
# Bước 2: Khởi tạo TF-IDF Vectorizer và học từ vựng (Vocabulary Fit)
# Để tối ưu RAM, chúng ta fit TF-IDF bằng cách sử dụng generator đọc file theo từng chunk
print("Đang đọc dữ liệu để học từ vựng (Fit TF-IDF) trên toàn bộ dữ liệu...")
start_time = time.time()
tfidf = TfidfVectorizer(stop_words='english', max_features=10000)

csv_path = '../preprocessing/preprocessed_data.csv'
tfidf.fit(text_generator(csv_path))

print(f"Đã học xong từ vựng! Kích thước từ điển: {len(tfidf.vocabulary_)}")
print(f"Thời gian huấn luyện: {time.time() - start_time:.2f} giây.")

Đang đọc dữ liệu để học từ vựng (Fit TF-IDF) trên toàn bộ dữ liệu...


Đã học xong từ vựng! Kích thước từ điển: 10000
Thời gian huấn luyện: 158.47 giây.


In [4]:
# Bước 3: Tạo chỉ mục (indexing) các sản phẩm duy nhất và chuyển đổi sang TF-IDF
print("Đang tạo chỉ mục cho các sản phẩm duy nhất...")
start_time = time.time()
seen_parent_asins = set()
unique_products = []

for chunk in pd.read_csv(csv_path, usecols=['parent_asin', 'title', 'main_category', 'content'], chunksize=100000):
    # Lọc nội dung tiếng Anh
    chunk['content'] = chunk['content'].fillna('')
    chunk = chunk[chunk['content'].apply(is_english)]
    
    # Loại bỏ trùng lặp trong chunk
    chunk = chunk.drop_duplicates(subset=['parent_asin'])
    
    # Lọc sản phẩm đã tồn tại
    chunk = chunk[~chunk['parent_asin'].isin(seen_parent_asins)]
    seen_parent_asins.update(chunk['parent_asin'])
    
    unique_products.append(chunk)

df_unique = pd.concat(unique_products, ignore_index=True)
print(f"Tổng số sản phẩm duy nhất: {len(df_unique)}")

# Biến đổi dữ liệu sang ma trận TF-IDF
tfidf_matrix = tfidf.transform(df_unique['content'])
print(f"Kích thước ma trận TF-IDF: {tfidf_matrix.shape}")
print(f"Thời gian tạo chỉ mục: {time.time() - start_time:.2f} giây.")

# Giải phóng bộ nhớ không cần thiết
del seen_parent_asins
gc.collect()

Đang tạo chỉ mục cho các sản phẩm duy nhất...


Tổng số sản phẩm duy nhất: 264322


Kích thước ma trận TF-IDF: (264322, 10000)
Thời gian tạo chỉ mục: 105.39 giây.


10

In [5]:
# Bước 4: Định nghĩa hàm gợi ý sản phẩm dựa trên văn bản nhập vào (Content Text)
def get_recommendations_by_text(query_text, top_k=10):
    """
    Gợi ý sản phẩm dựa trên văn bản mô tả đầu vào (content text).
    Đầu vào: content_text (str).
    Đầu ra: List 10 sản phẩm (mỗi sản phẩm là một dict chứa thông tin).
    """
    # 1. Chuyển đổi văn bản truy vấn đầu vào thành vector TF-IDF
    query_vector = tfidf.transform([query_text])
    
    # 2. Tính toán độ tương đồng Cosine với toàn bộ ma trận TF-IDF của sản phẩm duy nhất
    sim_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    # 3. Lấy ra top_k sản phẩm có độ tương đồng cao nhất
    top_indices = np.argsort(sim_scores)[::-1][:top_k]
    
    # 4. Tạo danh sách kết quả đầu ra
    results = []
    for idx in top_indices:
        row = df_unique.iloc[idx]
        results.append({
            'parent_asin': row['parent_asin'],
            'title': row['title'],
            'main_category': row['main_category'],
            'similarity_score': float(sim_scores[idx])
        })
        
    return results

In [6]:
# Bước 5: Chạy thử nghiệm gợi ý với đầu vào là văn bản mô tả (text) và in ra danh sách 10 sản phẩm
query = "black leather protective wallet case cover with card holder slot for iPhone"
print(f"Đầu vào: '{query}'\n")

print("--- DANH SÁCH 10 SẢN PHẨM GỢI Ý ---")
recommendations = get_recommendations_by_text(query, top_k=10)

# In kết quả dạng list
import pprint
pprint.pprint(recommendations)

Đầu vào: 'black leather protective wallet case cover with card holder slot for iPhone'

--- DANH SÁCH 10 SẢN PHẨM GỢI Ý ---


[{'main_category': 'Cell Phones & Accessories',
  'parent_asin': 'B00MXLJ370',
  'similarity_score': 0.5853271938650182,
  'title': 'iPhone 5 case,case for iPhone 5,Kaseberry iPhone 5 Wallet Leather '
           'Case Stand with Credit ID Card Slot Holder Cover Pouch for iPhone '
           '5 5S'},
 {'main_category': 'Cell Phones & Accessories',
  'parent_asin': 'B009LZQU8Q',
  'similarity_score': 0.5845040127117888,
  'title': 'Wallet Crocodile Leather Case Cover for Samsung Galaxy Nexus I9250 '
           'Black'},
 {'main_category': 'Cell Phones & Accessories',
  'parent_asin': 'B07JGR3Q12',
  'similarity_score': 0.5636229481583982,
  'title': 'iPhone 8 Plus Phone Case, ALYEE Magnetic Detachable Premium PU '
           'Leather Wallet Phone Case Flip Cover Protective Case for iPhone '
           '8/7 Plus with Credit Card Slot & Cash Pockets, Gray'},
 {'main_category': 'Cell Phones & Accessories',
  'parent_asin': 'B096Z6TZS8',
  'similarity_score': 0.5430184934810685,
  'title': '